---

<div align="center">
  <img src="https://raw.githubusercontent.com/devicons/devicon/master/icons/python/python-original.svg" width="80"/>
</div>

<h1 align="center">GenAI & Advanced Nets: O Mecanismo de Atenção </h1>

<h3 align="center">PhD. Julles Mitoura</h3>

<div align="center">
  <img src="https://img.shields.io/badge/Python-3776AB?style=for-the-badge&logo=python&logoColor=white"/>
  <img src="https://img.shields.io/badge/Jupyter-F37626?style=for-the-badge&logo=jupyter&logoColor=white"/>
</div>

---

<div align="center">

## <span style="color:#1E90FF;">Uso do Modelo</span>

</div>

Este notebook valida o modelo treinado em `genai_aula1_attention.ipynb` com foco em **inferência** e **leitura dos resultados**.

Fluxo da análise:
1. Carregar o checkpoint e reconstruir a arquitetura.
2. Executar predição do próximo token.
3. Exibir ranking `top-k` com probabilidades.
4. Interpretar se o token esperado aparece entre as melhores hipóteses.

In [1]:
import torch
from torch import nn

# mesma arquitetura usada no notebook de treino
class Transformer(nn.Module):
    def __init__(self, vocab_size, embedding_dim, n_heads, n_layers, dropout):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.attention_layers = nn.ModuleList([
            nn.MultiheadAttention(embedding_dim, n_heads, dropout=dropout, batch_first=True)
            for _ in range(n_layers)
        ])
        self.feed_forward = nn.Sequential(
            nn.Linear(embedding_dim, embedding_dim),
            nn.ReLU(),
            nn.Linear(embedding_dim, embedding_dim)
        )
        self.out = nn.Linear(embedding_dim, vocab_size)

    def forward(self, X):
        x = self.embedding(X)
        for attention in self.attention_layers:
            attn_out, _ = attention(x, x, x)
            x = x + attn_out
        x = self.feed_forward(x)
        return self.out(x)

# configuração de execução
torch.manual_seed(42)
device = "cuda" if torch.cuda.is_available() else "cpu"

# carrega checkpoint salvo
checkpoint_path = "models/transformer_attention.pt"
ckpt = torch.load(checkpoint_path, map_location=device)

model = Transformer(
    vocab_size=ckpt["vocab_size"],
    embedding_dim=ckpt["embedding_dim"],
    n_heads=ckpt["n_heads"],
    n_layers=ckpt["n_layers"],
    dropout=ckpt["dropout"],
).to(device)
model.load_state_dict(ckpt["model_state_dict"])
model.eval()

print("Checkpoint carregado com sucesso!")
print(f"Arquivo : {checkpoint_path}")
print(f"Device  : {device}")
print("Hiperparâmetros do modelo:")
print(f"- vocab_size   : {ckpt['vocab_size']}")
print(f"- embedding_dim: {ckpt['embedding_dim']}")
print(f"- n_heads      : {ckpt['n_heads']}")
print(f"- n_layers     : {ckpt['n_layers']}")
print(f"- dropout      : {ckpt['dropout']}")

Checkpoint carregado com sucesso!
Arquivo : models/transformer_attention.pt
Device  : cpu
Hiperparâmetros do modelo:
- vocab_size   : 1000
- embedding_dim: 64
- n_heads      : 8
- n_layers     : 2
- dropout      : 0.1


In [2]:
# funções utilitárias para inferência e apresentação

def predict_next_token(model, x_input, top_k=5):
    model.eval()
    x_input = x_input.to(device)

    with torch.no_grad():
        logits = model(x_input)
        last_logits = logits[:, -1, :]
        probs = torch.softmax(last_logits, dim=-1)

        pred_top1 = probs.argmax(dim=-1).item()
        top_probs, top_tokens = torch.topk(probs, k=top_k, dim=-1)

    return pred_top1, top_tokens[0].tolist(), top_probs[0].tolist()


def show_topk(top_tokens, top_probs, expected_token=None):
    print("\nTop-k tokens mais prováveis:")
    print("token\tprobabilidade")
    print("-----\t-------------")

    for token, prob in zip(top_tokens, top_probs):
        marker = " <== esperado" if expected_token is not None and token == expected_token else ""
        print(f"{token}\t{prob:.4%}{marker}")


# exemplo de inferência
# use tokens no intervalo [0, vocab_size-1]
x_test = torch.tensor(
    [[356, 69, 541, 586, 192, 404, 423, 338, 989, 757, 639, 914, 907, 456, 401, 663]],
    dtype=torch.long,
)
expected_next_token = 123

pred_top1, top_tokens, top_probs = predict_next_token(model, x_test, top_k=5)

print("Entrada:", x_test.squeeze(0).tolist())
print(f"Token esperado      : {expected_next_token}")
print(f"Predição top-1      : {pred_top1}")
print(f"Acerto top-1        : {'sim' if pred_top1 == expected_next_token else 'não'}")
print(f"Esperado está no top-5: {'sim' if expected_next_token in top_tokens else 'não'}")

show_topk(top_tokens, top_probs, expected_token=expected_next_token)

Entrada: [356, 69, 541, 586, 192, 404, 423, 338, 989, 757, 639, 914, 907, 456, 401, 663]
Token esperado      : 123
Predição top-1      : 123
Acerto top-1        : sim
Esperado está no top-5: sim

Top-k tokens mais prováveis:
token	probabilidade
-----	-------------
123	99.9992% <== esperado
594	0.0004%
687	0.0001%
287	0.0001%
156	0.0001%


---

### Interpretação rápida

- **Predição top-1**: token de maior probabilidade previsto pelo modelo.
- **Top-5**: mostra alternativas plausíveis quando o top-1 não coincide com o esperado.
- Quando o token esperado aparece no **top-5**, o modelo ainda demonstra boa qualidade de ranking.

Com isso, o notebook de teste deixa de ser apenas uma chamada simples de inferência e passa a oferecer uma leitura mais completa do comportamento do modelo.